# RQ7 Model Performance for Obesity Prediction

**Research question:** Which machine learning model performs best in predicting obesity classification?

This Kaggle notebook loads the raw dataset, generates the publication-ready table as CSV, saves the figure as PDF, and prints the evidence-based answer.

In [ ]:

# ============================================================
# Kaggle-ready setup: load raw obesity dataset and shared helpers
# ============================================================
import os, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import chi2_contingency, pearsonr, spearmanr, f_oneway

# Kaggle output directory
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('outputs')
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Change this only if needed. On Kaggle, the dataset usually lives under /kaggle/input/...
DATASET_PATH = None


def find_dataset_file():
    """Find the obesity dataset on Kaggle or locally. Supports CSV and Excel."""
    if DATASET_PATH:
        return DATASET_PATH
    patterns = [
        '/kaggle/input/**/*.csv', '/kaggle/input/**/*.xlsx', '/kaggle/input/**/*.xls',
        './*.csv', './*.xlsx', './*.xls',
        '../input/**/*.csv', '../input/**/*.xlsx', '../input/**/*.xls'
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(glob.glob(pattern, recursive=True))
    # Prefer files whose name looks like the obesity dataset
    ranked = sorted(candidates, key=lambda p: ('obesity' not in os.path.basename(p).lower(), len(p)))
    if not ranked:
        raise FileNotFoundError('No CSV/XLSX dataset file found. Upload the raw dataset to Kaggle Input or set DATASET_PATH manually.')
    return ranked[0]


def load_dataset():
    path = find_dataset_file()
    print(f'Loading dataset from: {path}')
    if path.lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
    return df


def clean_dataset(df):
    """Basic cleaning only; avoids changing the research meaning of variables."""
    df = df.copy()
    df = df.drop_duplicates().reset_index(drop=True)
    # Normalize object columns
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    return df


def add_obesity_group(df):
    """Collapse detailed target classes into paper-friendly groups."""
    df = df.copy()
    mapping = {
        'Insufficient_Weight': 'Underweight',
        'Normal_Weight': 'Normal',
        'Overweight_Level_I': 'Overweight',
        'Overweight_Level_II': 'Overweight',
        'Obesity_Type_I': 'Obese',
        'Obesity_Type_II': 'Obese',
        'Obesity_Type_III': 'Obese'
    }
    df['Obesity_Group'] = df['NObeyesdad'].map(mapping).fillna(df['NObeyesdad'])
    order = ['Underweight', 'Normal', 'Overweight', 'Obese']
    df['Obesity_Group'] = pd.Categorical(df['Obesity_Group'], categories=order, ordered=True)
    return df


def add_obesity_score(df):
    """Ordinal score for correlation/ANOVA summaries."""
    df = df.copy()
    score_map = {
        'Insufficient_Weight': 0,
        'Normal_Weight': 1,
        'Overweight_Level_I': 2,
        'Overweight_Level_II': 3,
        'Obesity_Type_I': 4,
        'Obesity_Type_II': 5,
        'Obesity_Type_III': 6
    }
    df['Obesity_Score'] = df['NObeyesdad'].map(score_map)
    return df


def save_table(df, filename):
    path = TABLE_DIR / filename
    df.to_csv(path, index=False)
    print(f'Saved table: {path}')
    return path


def save_figure(filename):
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, format='pdf', bbox_inches='tight')
    print(f'Saved figure: {path}')
    plt.show()
    return path


def percent_yes(series):
    return (series.astype(str).str.lower().eq('yes').mean() * 100)


def style_plot(title, xlabel=None, ylabel=None):
    plt.title(title, fontsize=14, weight='bold')
    if xlabel: plt.xlabel(xlabel, fontsize=11)
    if ylabel: plt.ylabel(ylabel, fontsize=11)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.grid(axis='y', alpha=0.25)
    for spine in ['top', 'right']:
        plt.gca().spines[spine].set_visible(False)


df = add_obesity_score(add_obesity_group(clean_dataset(load_dataset())))
display(df.head())
print(df['NObeyesdad'].value_counts())


## Analysis, table, figure, and answer

In [ ]:

# RQ7: Best machine learning model for obesity prediction
# Variables: all predictors except NObeyesdad

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X = df.drop(columns=['NObeyesdad', 'Obesity_Group', 'Obesity_Score'])
y = df['NObeyesdad']

categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = X.select_dtypes(exclude='object').columns.tolist()

preprocess_scaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)
preprocess_unscaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

models = {
    'Logistic Regression': (preprocess_scaled, LogisticRegression(max_iter=2000, class_weight='balanced')),
    'KNN': (preprocess_scaled, KNeighborsClassifier(n_neighbors=7)),
    'Decision Tree': (preprocess_unscaled, DecisionTreeClassifier(random_state=42, class_weight='balanced')),
    'Random Forest': (preprocess_unscaled, RandomForestClassifier(n_estimators=150, random_state=42, class_weight='balanced')),
    'Gradient Boosting': (preprocess_unscaled, GradientBoostingClassifier(random_state=42, n_estimators=80))
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

rows = []
predictions = {}
for name, (prep, clf) in models.items():
    pipe = Pipeline(steps=[('preprocess', prep), ('model', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    predictions[name] = y_pred
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'macro_precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_test, y_pred, average='macro', zero_division=0)
    })

performance = pd.DataFrame(rows).sort_values('macro_f1', ascending=False).round(5)
save_table(performance, 'RQ7_model_performance_comparison.csv')
display(performance)

best_model = performance.iloc[0]['model']
labels = sorted(y.unique())
cm = pd.DataFrame(confusion_matrix(y_test, predictions[best_model], labels=labels), index=[f'Actual: {l}' for l in labels], columns=[f'Pred: {l}' for l in labels])
cm_out = cm.reset_index().rename(columns={'index':'actual_class'})
save_table(cm_out, 'RQ7_best_model_confusion_matrix.csv')
display(cm)

# Figure: grouped bar chart for model metrics
fig_df = performance.set_index('model')[['accuracy', 'macro_precision', 'macro_recall', 'macro_f1']]
ax = fig_df.plot(kind='bar', figsize=(10, 5.8), width=0.8)
style_plot('Machine Learning Model Performance for Obesity Prediction', 'Model', 'Score')
plt.ylim(0, 1.05)
plt.xticks(rotation=25, ha='right')
plt.legend(frameon=False, fontsize=9, ncol=2)
save_figure('RQ7_model_performance_comparison.pdf')

# Extra figure: confusion matrix for best model
plt.figure(figsize=(8, 6))
plt.imshow(cm.values, aspect='auto')
plt.colorbar(label='Count')
plt.xticks(np.arange(len(labels)), labels, rotation=45, ha='right')
plt.yticks(np.arange(len(labels)), labels)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm.values[i, j], ha='center', va='center')
style_plot(f'Confusion Matrix: {best_model}', 'Predicted class', 'Actual class')
save_figure('RQ7_best_model_confusion_matrix.pdf')

print('\nANSWER TO RQ7')
print(f"The best-performing model is {best_model}, selected by highest macro-F1 score.")
print(f"Best macro-F1: {performance.iloc[0]['macro_f1']:.3f}; accuracy: {performance.iloc[0]['accuracy']:.3f}.")
print('Macro-F1 is preferred because the target has multiple classes and class balance should be considered.')
